# Cascade Denoised Smoothing on Kaggle (CIFAR-10)
Settings: **Accelerator = GPU T4 x2**, **Internet = On**, Persistence = *Files only* (if available).

Each phase can be run in its own session/commit. Outputs go to `/kaggle/working/dds/out`.
To resume after a session ends: add this notebook's previous **Output** as an input and set `PREV_OUT` below.

**Save & Run All (Commit)**: set `PHASES` in the config cell first. Typical commits:
1. `["probe", "record", "analyze"]` with `MAX_IMAGES=500` (record stops after `TIME_BUDGET` h).
2. If not finished: add this notebook's Output as input, set `PREV_OUT`, `PHASES=["record","analyze"]`, commit again.

In [ ]:
# ---------------- config ----------------
SIGMA      = 0.25
SKIP       = 20          # certify images idx % 20 == 0 (500 imgs); probe uses the other 9,500
MAX_IMAGES = 500         # start with 100, later raise to 500 (runs resume, nothing is recomputed)
N          = 10000
BATCH      = 500         # lower to 250 on OOM
AMP        = True        # fp16 on the expensive path; checked in cell 3; SAME flag for dds & cascade
DELTA, CAL_ALPHA = 0.01, 0.05
# Which steps to run. "Save & Run All (Commit)" runs every cell, so only listed phases do work:
#   sanity | probe | record | analyze | cheapmodels | profile | candidates | direct (old per-mode certify, slow)
PHASES      = ["probe", "record", "analyze"]
TIME_BUDGET = 10.5       # hours for record() per commit; Kaggle kills sessions at 12 h
PREV_OUT   = "/kaggle/input/notebooks/trncqun/diffpure/dds/out"
# phase "cheapmodels": stronger noise-trained cheap models (public archives, only this sigma is extracted)
CHEAP_SOURCES = {
    # ACES (Horvath et al.): Consistency-trained ResNet-110, 2.3 GB tar.gz streamed
    "aces": ("https://files.sri.inf.ethz.ch/aces/aces-models.tar.gz",
             ["*cifar10/consistency/noise_{SIG}/checkpoint.pth.tar"]),
    # smoothing-ensembles (Horvath et al.): 3 Consistency ResNet-110 per sigma, 5.0 GB zip
    "ens": ("https://files.sri.inf.ethz.ch/smoothing-ensembles/smoothing-ensemble-models.zip",
            ["*models-cifar10/consistency/resnet110/{SIG}/*.pth.tar"]),
}
TOPK = 3        # e.g. "/kaggle/input/<this-notebook-output>/dds/out" to resume
REPO       = "https://github.com/ducquan19/dds.git"
GIT_REF    = "main"      # pin a commit sha for a whole experiment (e.g. "a1b2c3d") so resumed runs use identical code
COHEN_GDRIVE_ID = "1h_TpbXm5haY5f-l4--IKylmdz6tvPoR4"   # models.tar from locuslab/smoothing README

import os, subprocess, torch
SIG = f"{SIGMA:.2f}"
AMPF = "--amp" if AMP else ""
NGPU = torch.cuda.device_count()
print("torch", torch.__version__, "| GPUs:", [torch.cuda.get_device_name(i) for i in range(NGPU)])

In [ ]:
# ---------------- 1. setup (~5 min) ----------------
%cd /kaggle/working
if not os.path.exists("dds"):
    !git clone -q {REPO} dds
%cd /kaggle/working/dds
!git fetch -q origin && git checkout -q {GIT_REF} && (git pull -q origin {GIT_REF} 2>/dev/null || true)
SHA = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip()
print("code version:", SHA)
!pip install -q gdown statsmodels "transformers>=4.40,<5"

if not os.path.exists("cifar10/cifar10_uncond_50M_500K.pt"):
    !wget -q -O cifar10/cifar10_uncond_50M_500K.pt https://openaipublic.blob.core.windows.net/diffusion/march-2021/cifar10_uncond_50M_500K.pt
if not os.path.exists("cifar10/archs"):
    !git clone -q --depth 1 https://github.com/locuslab/smoothing.git /tmp/cohen && cp -r /tmp/cohen/code/archs cifar10/archs
if not os.path.exists("models/cifar10/resnet110/noise_0.25/checkpoint.pth.tar"):
    !gdown -q {COHEN_GDRIVE_ID} -O /tmp/models.tar && mkdir -p /tmp/cm && tar -xf /tmp/models.tar -C /tmp/cm && rm /tmp/models.tar
    !mkdir -p models && cp -r "$(dirname "$(dirname "$(find /tmp/cm -type d -path '*cifar10/resnet110/noise_0.25' | head -1)")")" models/cifar10 && rm -rf /tmp/cm
!ls models/cifar10/resnet110
os.makedirs("out", exist_ok=True); os.makedirs("logs", exist_ok=True)
prev_sha = open("out/git_sha.txt").read().strip() if os.path.exists("out/git_sha.txt") else None
if PREV_OUT:
    if PREV_OUT.endswith((".tgz", ".tar.gz")):          # archive written by the last cell of a previous run
        !tar -xzf {PREV_OUT} --skip-old-files -C .
    else:                                              # an out/ directory (notebook output or dataset)
        !cp -rn {PREV_OUT}/. out/
    !ls out
    prev_sha = open("out/git_sha.txt").read().strip() if os.path.exists("out/git_sha.txt") else None
if prev_sha and prev_sha != SHA:
    print(f"WARNING: out/ was produced with code {prev_sha}, now running {SHA}. Set GIT_REF = '{prev_sha}' to resume consistently.")
open("out/git_sha.txt", "w").write(SHA)
import sys; sys.path.insert(0, "cifar10")
from torchvision import datasets
datasets.CIFAR10("data/cifar10", train=False, download=True)
CKPT = f"models/cifar10/resnet110/noise_{SIG}/checkpoint.pth.tar"

In [ ]:
# ---------------- 2. sanity: imports, checkpoint, AMP vs FP32 agreement (~3 min) ----------------
if "sanity" in PHASES:
    import core                      # fails if the binom_test fix is missing
    from DRM_cascade import CascadeRobustModel
    m = CascadeRobustModel(SIGMA, mode="cascade", tau=1.01, cheap_ckpt=CKPT)
    ds = datasets.CIFAR10("data/cifar10", train=False, transform=__import__("torchvision").transforms.ToTensor())
    x = torch.stack([ds[i][0] for i in range(1, 200, 2)]).cuda().repeat_interleave(5, 0)   # not certify images
    torch.manual_seed(0); xn = x + torch.randn_like(x) * SIGMA
    m.amp = False; p32 = torch.cat([m.expensive_logits(c).argmax(1) for c in xn.split(250)])
    m.amp = True;  p16 = torch.cat([m.expensive_logits(c).argmax(1) for c in xn.split(250)])
    y = torch.tensor([ds[i][1] for i in range(1, 200, 2)]).cuda().repeat_interleave(5)
    print(f"AMP/FP32 agreement {(p32 == p16).float().mean():.4f} | dds acc fp32 {(p32 == y).float().mean():.3f} amp {(p16 == y).float().mean():.3f}")
    print(f"cheap acc {(m.cheap_logits(xn).argmax(1) == y).float().mean():.3f}")
    # timing of one certify-sized pass (N0+N samples) on one GPU
    import time
    for amp in [False, True]:
        m.amp = amp; torch.cuda.synchronize(); t0 = time.time()
        for _ in range((N + 100) // BATCH): m.expensive_logits(x[:1].repeat(BATCH, 1, 1, 1) + torch.randn(BATCH, 3, 32, 32, device="cuda") * SIGMA)
        torch.cuda.synchronize(); print(f"amp={amp}: dds ~{time.time() - t0:.1f} s/img on 1 GPU")
    del m; torch.cuda.empty_cache()
    # -> if agreement < 0.99, set AMP = False in the config cell

In [ ]:
# helper: run certify_cascade on all GPUs in parallel (one shard per GPU), resumable
def certify(mode, tau=None, tag=None):
    tag = tag or mode
    procs = []
    for s in range(NGPU):
        out = f"out/{tag}_{SIG}_s{s}.tsv"
        cmd = (f"CUDA_VISIBLE_DEVICES={s} PYTHONPATH=cifar10 python cifar10/certify_cascade.py --mode {mode} "
               f"--sigma {SIGMA} --cheap_ckpt {CKPT} --skip {SKIP} --max_images {MAX_IMAGES} --N {N} "
               f"--alpha 0.0005 --batch_size {BATCH} --shard {s} --num_shards {NGPU} --resume "
               f"--outfile {out} {AMPF} {'' if tau is None else '--tau ' + str(tau)} "
               f"> logs/{tag}_{SIG}_s{s}.log 2>&1")
        procs.append(subprocess.Popen(cmd, shell=True))
    rc = [p.wait() for p in procs]
    !tail -n 2 logs/{tag}_{SIG}_s*.log
    assert all(r == 0 for r in rc), f"{tag} failed, see logs/"

def report(*extra):
    files = [f"{t}=out/{t}_{SIG}_s*.tsv" for t in ["dds", "cheap"] + list(extra)]
    files = [f for f in files if __import__("glob").glob(f.split("=")[1])]
    fs = " ".join(files)
    !PYTHONPATH=cifar10 python cifar10/compare.py {fs} --intersect

In [ ]:
# ---------------- 3. probe + calibrate tau  (feasibility test, ~15-30 min) ----------------
if "probe" in PHASES:
    !PYTHONPATH=cifar10 python cifar10/probe.py --sigma {SIGMA} --cheap_ckpt {CKPT} --num_images 9500 --K 4 \
        --split test_holdout --certify_skip {SKIP} --batch_size {BATCH} {AMPF} --out out/probe_{SIG}.npz
    !PYTHONPATH=cifar10 python cifar10/calibrate_tau.py out/probe_{SIG}.npz --delta {DELTA} --alpha {CAL_ALPHA} --out out/tau_{SIG}.txt
    TAU = float(open(f"out/tau_{SIG}.txt").read())
    print("calibrated tau =", TAU)
    # DECISION: accept rate at this tau ~ fraction of diffusion calls saved. < ~0.5  => speed-up < 2x.

In [ ]:
# ---------------- 4. certify: cascade (calibrated tau) + cheap ----------------
if "direct" in PHASES:
    TAU = float(open(f"out/tau_{SIG}.txt").read())
    certify("cascade", tau=TAU, tag="cascade_cal")
    certify("cheap")
    report("cascade_cal")

In [ ]:
# ---------------- 5. certify: dds baseline (most expensive; resumable) ----------------
if "direct" in PHASES:
    certify("dds")
    report("cascade_cal")

In [ ]:
# ---------------- 6. tau sweep for the Pareto plot (optional, run when budget allows) ----------------
if "direct" in PHASES:
    for t in [0.5, 0.7, 0.9, 0.99]:
        certify("cascade", tau=t, tag=f"cascade_tau{t}")
    report("cascade_cal", *[f"cascade_tau{t}" for t in [0.5, 0.7, 0.9, 0.99]])

## Phase 2 — raise the speed-up (after the 100-image run)
Cost model observed: `sec/img(cascade) ≈ cheap + exp_frac × dds`. Two levers: make `cheap` ~free, and lower `exp_frac`
with a stronger noise-trained cheap model. Each candidate only needs **probe + calibrate** (~30 min), no certify.

In [ ]:
# ---------------- 9. profile: where does the time go? (~2 min) ----------------
if "profile" in PHASES:
    from DRM_cascade import CascadeRobustModel
    for amp in [False, True]:
        m = CascadeRobustModel(SIGMA, mode="cascade", tau=1.01, cheap_ckpt=CKPT, amp=amp)
        r = m.profile(batch=BATCH)
        print(f"amp={amp}: sec per 10k samples (1 GPU) ->", {k: round(v, 2) for k, v in r.items()})
        del m; torch.cuda.empty_cache()
    # If vit >> unet, the bottleneck of one-step DDS on CIFAR is the ViT at 224px, not the diffusion model.

In [ ]:
# ---------------- 10. compare cheap models by accept rate at delta (probe + calibrate only) ----------------
if "candidates" in PHASES:
    # Download each checkpoint (Cohen format, CIFAR ResNet-110, trained at THIS sigma) and/or upload as a Kaggle Dataset.
    CANDIDATES = {
        "cohen":       CKPT,
        # "smoothadv":   "/kaggle/input/<dataset>/smoothadv_resnet110_0.25.pth.tar",
        # "consistency": "/kaggle/input/<dataset>/consistency_resnet110_0.25.pth.tar",
    }
    import numpy as np
    for name, ck in CANDIDATES.items():
        out = f"out/probe_{SIG}_{name}_amp{int(AMP)}.npz"
        if not os.path.exists(out):
            !PYTHONPATH=cifar10 python cifar10/probe.py --sigma {SIGMA} --cheap_ckpt {ck} --num_images 9500 --K 4 \
                --split test_holdout --certify_skip {SKIP} --batch_size {BATCH} {AMPF} --out {out} > logs/probe_{name}.log 2>&1
        print(f"==== {name}")
        !tail -n 7 logs/probe_{name}.log | head -n 1
        for d in [0.005, 0.01, 0.02]:
            !PYTHONPATH=cifar10 python cifar10/calibrate_tau.py {out} --delta {d} --alpha {CAL_ALPHA} | sed -n 2p | sed "s/^/delta={d}  /"
    # Pick the candidate with the highest accept rate at delta=0.01, write its tau, then rerun cell 6 with it:
    #   CKPT = CANDIDATES["<best>"]; and calibrate_tau.py ... --out out/tau_{SIG}.txt

## FAST PATH — record once, analyse every mode (recommended)
One GPU pass stores cheap + expensive predictions for all N0+N noise samples of each image
(~(t_cheap + t_exp) per image ≈ 55 s on one T4 with AMP → 500 images ≈ 3.7 h on T4 x2).
Then dds / cheap / cascade at ANY tau are certified on CPU in seconds. Replaces cells 6–8.
Needs: cells 1–2 (setup), and cell 5 (probe → `out/tau_<sigma>.txt`) run with the same `AMP` flag.

In [ ]:
# ---------------- R1. record (resumable: rerun after a session ends, finished images are skipped) ----------------
def record(ckpt=None, out_dir=None, only_cheap=False, norm="auto"):
    ckpt = ckpt or CKPT; out_dir = out_dir or f"out/rec_{SIG}"
    procs = []
    for s in range(NGPU):
        cmd = (f"CUDA_VISIBLE_DEVICES={s} PYTHONPATH=cifar10 python cifar10/record.py --sigma {SIGMA} "
               f"--cheap_ckpt {ckpt} --skip {SKIP} --max_images {MAX_IMAGES} --N {N} --batch_size {BATCH} "
               f"--shard {s} --num_shards {NGPU} {AMPF} --time_budget {TIME_BUDGET} {'--only_cheap' if only_cheap else ''} --cheap_norm {norm} "
               f"--out_dir {out_dir} > logs/record_{os.path.basename(out_dir)}_s{s}.log 2>&1")
        procs.append(subprocess.Popen(cmd, shell=True))
    rc = [p.wait() for p in procs]
    lg = f"logs/record_{os.path.basename(out_dir)}_s*.log"
    !tail -n 2 {lg}
    assert all(r == 0 for r in rc), "record failed, see logs/"

if "record" in PHASES:
    record()

In [ ]:
# ---------------- R2. analyse (CPU, seconds) ----------------
if "analyze" in PHASES:
    !PYTHONPATH=cifar10 python cifar10/analyze_record.py out/rec_{SIG} --tau_file out/tau_{SIG}.txt --paired --tsv_dir out/tsv --summary_csv out/summary_{SIG}_cohen.csv
    !PYTHONPATH=cifar10 python cifar10/plot_pareto.py cohen=out/summary_{SIG}_cohen.csv --radius 0.5 0.75 --out out/pareto_{SIG}
    from IPython.display import Image, display; display(Image(f"out/pareto_{SIG}.png"))

In [ ]:
# ---------------- R3. new cheap model on the SAME noise: only the cheap path is run (~t_cheap per image) ----------------
# NEW_CKPT = "/kaggle/input/<dataset>/smoothadv_resnet110_0.25.pth.tar"; NAME = "smoothadv"
# 1) calibrate its tau on held-out images (cell 10 or):
# !PYTHONPATH=cifar10 python cifar10/probe.py --sigma {SIGMA} --cheap_ckpt {NEW_CKPT} --num_images 9500 --K 4 --split test_holdout --certify_skip {SKIP} --batch_size {BATCH} {AMPF} --out out/probe_{SIG}_{NAME}.npz
# !PYTHONPATH=cifar10 python cifar10/calibrate_tau.py out/probe_{SIG}_{NAME}.npz --delta {DELTA} --alpha {CAL_ALPHA} --out out/tau_{SIG}_{NAME}.txt
# 2) record its cheap path and analyse against the stored expensive predictions:
# record(NEW_CKPT, f"out/rec_{SIG}_{NAME}", only_cheap=True)
# !PYTHONPATH=cifar10 python cifar10/analyze_record.py out/rec_{SIG} --cheap_rec out/rec_{SIG}_{NAME} --tau_file out/tau_{SIG}_{NAME}.txt --paired --summary_csv out/summary_{SIG}_{NAME}.csv
# !PYTHONPATH=cifar10 python cifar10/plot_pareto.py cohen=out/summary_{SIG}_cohen.csv {NAME}=out/summary_{SIG}_{NAME}.csv --radius 0.5 0.75 --out out/pareto_{SIG}_all

## S. Stronger cheap models (phase `cheapmodels`, ~1-1.5 h)
1. stream the public ACES / smoothing-ensembles archives and extract only Consistency ResNet-110 checkpoints at this sigma;
2. base probe stores DDS predictions once; each candidate (both normalisations) is screened with the cheap path only
   on calibration **half 0**; the wrong normalisation shows up as collapsed accuracy;
3. the final tau of every chosen model is calibrated on **half 1** (disjoint from the selection), then the best `TOPK`
   are recorded on the certification noise and analysed against DDS together with Cohen.
Needs `out/rec_<sigma>/` from the record phase (`PREV_OUT`).

In [ ]:
# ---------------- S. Stronger cheap models: download -> screen (half 0) -> calibrate (half 1) -> record TOPK ----------------
import glob, re, subprocess, numpy as np, pandas as pd

def sh(cmd, check=True):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if check and r.returncode:
        print(r.stdout[-2000:], r.stderr[-3000:]); raise RuntimeError(cmd)
    return r.stdout if check else r.stdout + r.stderr

def cal(pr, half, out=None):
    o = sh(f"PYTHONPATH=cifar10 python cifar10/calibrate_tau.py {pr} --delta {DELTA} --alpha {CAL_ALPHA} --half {half}"
           + (f" --out {out}" if out else ""))
    m = re.search(r"chosen tau=([\d.]+) \| accept rate=([\d.]+)", o)
    return (float(m.group(1)), float(m.group(2))) if m else (None, 0.0)

if "cheapmodels" in PHASES:
    assert glob.glob(f"out/rec_{SIG}/*.npz"), "need the DDS records out/rec_<sigma>/ (set PREV_OUT)"
    # 1) download ONLY the checkpoints for this sigma
    for src, (url, pats) in CHEAP_SOURCES.items():
        dst = f"/tmp/cheap/{src}"
        if not glob.glob(f"{dst}/**/*.pth.tar", recursive=True):
            os.makedirs(dst, exist_ok=True)
            pp = " ".join(f"'{p.format(SIG=SIG)}'" for p in pats)
            if url.endswith((".tar.gz", ".tgz")):        # stream: never stores the whole archive
                print(sh(f"set -o pipefail; wget -qO- {url} | tar -xz -C {dst} --wildcards {pp}", check=False)[-500:])
            else:                                         # zip has to be downloaded first
                sh(f"wget -q -O /tmp/cheap/{src}.zip {url}")
                print(sh(f"cd {dst} && unzip -qo /tmp/cheap/{src}.zip {pp}; rm -f /tmp/cheap/{src}.zip", check=False)[-500:])
        print(f"[{src}]\n" + sh(f"find {dst} -name '*.pth.tar'"))
    cands = {re.sub(r"[^A-Za-z0-9]+", "_", f"{s}_{os.path.relpath(c, f'/tmp/cheap/{s}')}".replace(".pth.tar", "")): c
             for s in CHEAP_SOURCES for c in sorted(glob.glob(f"/tmp/cheap/{s}/**/*.pth.tar", recursive=True))}
    print(len(cands), "candidate checkpoints")

    # 2) base probe (stores DDS predictions), then cheap-only screening on calibration half 0
    os.makedirs("out/cand", exist_ok=True)
    base = f"out/probe_{SIG}_base_v2.npz"   # v2: image sampling independent of model loading
    P = (f"PYTHONPATH=cifar10 python cifar10/probe.py --sigma {SIGMA} --num_images 9500 --K 4 --split test_holdout "
         f"--certify_skip {SKIP} --batch_size {BATCH} {AMPF}")
    if not os.path.exists(base):
        sh(f"{P} --cheap_ckpt {CKPT} --out {base}")

    def screen(name, ckpt, norm):
        pr = f"out/cand/probe_{SIG}_{name}.npz"
        if not os.path.exists(pr):
            r = subprocess.run(f"{P} --cheap_ckpt {ckpt} --cheap_norm {norm} --reuse_exp {base} --out {pr}",
                               shell=True, capture_output=True, text=True)
            if r.returncode:
                if name == "cohen":   # known-good model: a failure here is a pipeline bug, show everything
                    print(r.stdout[-1500:]); print(r.stderr[-4000:])
                    raise RuntimeError("screening the Cohen model failed - see the error above")
                print(f"  skip {name}: {(r.stderr.strip().splitlines() or ['?'])[-1][:200]}"); return None
        tau0, acc0 = cal(pr, 0)
        d = np.load(pr); f = d["first"]; h = np.arange(f.sum()) % 2 == 0
        ch, lb, ex = d["cheap"][f][h], d["label"][f][h], d["exp"][f][h]
        return dict(name=name, ckpt=ckpt, norm=norm, probe=pr, accept_h0=acc0,
                    cheap_acc=float(np.mean(ch == lb)), agree=float(np.mean(ch == ex)))

    rows = [screen("cohen", CKPT, "normalize")]
    for name, c in cands.items():
        res = [r for r in (screen(f"{name}_{nm}", c, nm) for nm in ("normalize", "center")) if r]
        for r in res:
            print(f"  {r['name']:<60} cheap_acc {r['cheap_acc']:.3f}  accept {r['accept_h0']:.3f}")
        if res:   # the wrong normalisation collapses accuracy -> keep the better one
            rows.append(max(res, key=lambda r: r["cheap_acc"]))
    rows = [r for r in rows if r]
    if len(rows) < 2:
        raise RuntimeError("no candidate checkpoint could be screened - see the 'skip' lines above")
    tab = pd.DataFrame(rows).sort_values("accept_h0", ascending=False)
    tab.to_csv(f"out/cand/screen_{SIG}.csv", index=False)
    print(tab[["name", "norm", "accept_h0", "cheap_acc", "agree"]].to_string(index=False))

    # 3) final tau on calibration half 1 (disjoint from selection), record TOPK + cohen, analyse, plot
    chosen = [r for _, r in tab.iterrows() if r["name"] != "cohen"][:TOPK]
    curves = []
    for r in [tab[tab.name == "cohen"].iloc[0]] + chosen:
        tf = f"out/cand/tau1_{SIG}_{r['name']}.txt"
        tau1, acc1 = cal(r["probe"], 1, tf)
        print(f"== {r['name']}: final tau={tau1} accept(half 1)={acc1:.3f}")
        cheap_rec = ""
        if r["name"] != "cohen":
            rec = f"out/rec_{SIG}_{r['name']}"
            record(r["ckpt"], rec, only_cheap=True, norm=r["norm"])
            cheap_rec = f"--cheap_rec {rec}"
        summ = f"out/summary_{SIG}_{r['name']}_h1.csv"
        print(sh(f"PYTHONPATH=cifar10 python cifar10/analyze_record.py out/rec_{SIG} {cheap_rec} --tau_file {tf} "
                 f"--paired --summary_csv {summ}"))
        curves.append(f"{r['name'][:22]}={summ}")
    sh(f"PYTHONPATH=cifar10 python cifar10/plot_pareto.py {' '.join(curves[:4])} --radius 0.5 0.75 --out out/pareto_{SIG}_cheapmodels")
    from IPython.display import Image, display; display(Image(f"out/pareto_{SIG}_cheapmodels.png"))


In [ ]:
# ---------------- Z. always last: compact archive of results (~20-30 MB) at the top of the Output ----------------
!tar -czf /kaggle/working/out_{SIG}.tgz out && ls -lh /kaggle/working/out_{SIG}.tgz
# Resume elsewhere: download this file (Output tab) -> upload as a Kaggle Dataset -> PREV_OUT = "/kaggle/input/<dataset>/out_0.25.tgz"